In [ ]:
# git clone https://github.com/GuangyuWangLab2021/Loki.git
# conda create -n loki_env python=3.9
# conda activate loki_env
# cd ./Loki/src
# pip install .

In [ ]:
from common.data_processing import data_preprocessing_ST1K
import torch
from scipy.sparse import issparse
import cv2
import os
import pandas as pd
import numpy as np
import scanpy as sc
from PIL import Image
import loki.utils
import loki.preprocess
import loki.predex
sc.settings.set_figure_params(dpi=80, facecolor="white")

In [ ]:
model_path = os.path.join('/path/to/checkpoint.pt') # The pretrained weights are avaliable on https://huggingface.co/WangGuangyuLab/Loki
device = 'cuda:0'
model, preprocess, tokenizer = loki.utils.load_model(model_path, device)
model.eval()

In [ ]:

class CLIPDataset(torch.utils.data.Dataset):
    def __init__(self, image_path, ST_adata):
        super(CLIPDataset, self).__init__()
        self.spatial_pos_csv =  ST_adata.obsm['spatial'].astype(int)
        self.block_r = 112

        if issparse(ST_adata.X):
            self.expression = ST_adata.X.todense()
        else:
            self.expression = ST_adata.X        

        self.whole_image = Image.open(image_path)
        self.whole_image = np.array(self.whole_image)
        self.whole_image = self.whole_image[:,:,0:3]

    def transform(self, image):
        w = image.shape[0]
        dim = (224, 224)
        if w>=224:
            image = cv2.resize(image, dim, interpolation = cv2.INTER_AREA)
        else:
            image = cv2.resize(image, dim, interpolation = cv2.INTER_CUBIC)

        return image

    def __getitem__(self, idx):
        item = {}
        n_patches=self.spatial_pos_csv.shape[0]
        image_patches = torch.zeros((n_patches,224, 224,3)).float()
        h=self.whole_image.shape[0]
        w=self.whole_image.shape[1]
        for i in range(n_patches):
            v1 = self.spatial_pos_csv[i,0]
            v2 = self.spatial_pos_csv[i,1]

            image = self.whole_image[max((v2-self.block_r),0):min((v2+self.block_r),h),max((v1-self.block_r),0):min((v1+self.block_r),w),:]                   
            image = self.transform(image)

            image_patches[i]=torch.tensor(image).float() #[N,224,224,3]
        item['expression'] = torch.tensor(self.expression).float()  #[N x features] (3467)

        item['image'] = image_patches
        items = {'meta': item}
        return items

    def __len__(self):
        return len([1])

In [ ]:
input_path='./data/HEST/COLON-CANCER-HEALTHY/'
adata_list, HE_image_paths,nuclei_mask_paths=data_preprocessing_ST1K(input_path)
names = [d[:-5] for d in os.listdir(input_path) if d.endswith('.h5ad')]

print(names)
print(len(names))
print(len(adata_list))
print(len(HE_image_paths))

In [ ]:
import pickle
with open(input_path+'my_predict_gene_2468.pkl', 'rb') as f:
    gene_list = pickle.load(f)
print(len(gene_list))
for i in range(len(adata_list)):
    adata_list[i] = adata_list[i][:,gene_list]

In [ ]:
import torch.nn.functional as F
def encode_images(
    model: torch.nn.Module,
    preprocess: callable,
    image_arrays,
    device
) -> torch.Tensor:
    image_embeddings = []

    for image in image_arrays:
        image=Image.fromarray(np.uint8(image.numpy()))  # Convert numpy array to PIL Image
        image_input = torch.stack([preprocess(image)]).to(device)

        # Generate the image features without gradient tracking
        with torch.no_grad():
            image_features = model.encode_image(image_input)

        # Accumulate the feature embeddings in the list
        image_embeddings.append(image_features)

    # Convert the list of embeddings to a NumPy array, then to a PyTorch tensor
    image_embeddings = torch.cat(image_embeddings, dim=0)

    # Normalize all embeddings across the feature dimension (L2 normalization)
    image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)

    return image_embeddings

In [ ]:
from torch.utils.data import DataLoader
dataset_list=[]
for i in range(len(adata_list)):
    dataset = CLIPDataset(image_path = HE_image_paths[i],ST_adata = adata_list[i])
    dataset_list.append(dataset)

train_index=[i for i in range(len(adata_list))]
for i in range(len(adata_list)):
    test_index=[i]
    new_train_index = [item for item in train_index if item not in test_index]
    train_dataset = torch.utils.data.ConcatDataset([dataset_list[i] for i in new_train_index])
    test_dataset = torch.utils.data.ConcatDataset([dataset_list[i] for i in test_index])
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)
    train_img_list=[]
    train_exp_list=[]
    test_img_list=[]
    test_exp_list=[]
    for batch in train_loader:
        item = batch['meta']
        batch = {k: v.squeeze(0) for k, v in item.items()}
        img_array = batch['image']
        image_embeddings = encode_images(model, preprocess, img_array, device)
        expression = batch['expression']
        train_img_list.append(image_embeddings)
        train_exp_list.append(expression)
    for batch in test_loader:
        item = batch['meta']
        batch = {k: v.squeeze(0) for k, v in item.items()}
        img_array = batch['image']
        image_embeddings = encode_images(model, preprocess, img_array, device)
        expression = batch['expression']
        test_img_list.append(image_embeddings)
        test_exp_list.append(expression)
    train_img = torch.cat(train_img_list, dim=0).cpu()
    train_exp = torch.cat(train_exp_list, dim=0).cpu()
    test_img = torch.cat(test_img_list, dim=0).cpu()
    print(f"Train image shape: {train_img.shape}, Train expression shape: {train_exp.shape}")
    print(f"Test image shape: {test_img.shape}")
    image_similarity = torch.matmul(test_img, train_img.T)
    predicted_expression = torch.matmul(image_similarity, train_exp)
    predicted_expression = predicted_expression.cpu().numpy()

    pred_adata = sc.AnnData(X=predicted_expression, obs=pd.DataFrame(index=range(predicted_expression.shape[0])), var=pd.DataFrame(index=gene_list))
    adata_path=os.path.join(input_path,names[test_index[0]],'results/OmiCLIP_pre.h5ad')
    pred_adata.write(adata_path)
    adata_list[i].write(os.path.join(input_path,names[test_index[0]],'results/OmiCLIP_gt.h5ad'))

    

